# Experimento 3. Ajuste de la PINN con puffs

`pinn_puff` (ADR 0005) aplica el núcleo de la red a la pluma de la última hora y a los puffs de las 24 h anteriores. Armar sus matrices toma unos 11 minutos por corrida, y las búsquedas de parámetros (`nox-ml tune`) son muchas corridas, así que **solo existen en la máquina del dueño del proyecto** (`runs/` no va en git). Este cuaderno lee lo que haya en `runs/` y, si falta algo, lo dice y muestra el comando que lo produce:

| Corrida | Qué es |
|---|---|
| `pinn_plume` | La PINN con pluma estacionaria (referencia) |
| `pinn_puff` | La PINN con puffs, con los parámetros por defecto |
| `pinn_puff.transport`, `pinn_puff.kernel` | Búsquedas con `nox-ml tune`: por su nombre, una sobre parámetros de transporte (que no reentrenan la red) y otra sobre parámetros del núcleo. El espacio que cada una exploró se lee de su `trials.csv` |
| `pinn_puff_k*` | Rejilla de la difusividad `kappa` (cada valor entrena su propia red) |

No hay nada que calcular aquí: solo funciones de lectura y gráficos de `nox_ml.validation`.

In [ ]:
from nox_ml.validation import compare, report
from nox_ml.validation import report_experiments as rx

RUNS = report.find_runs()
rx.runs_status(RUNS)

## 1. Las corridas simples

Error con estación oculta, cociente domingo ÷ hábil y placebo de los modelos PINN. Un modelo compite si su placebo está centrado en 1 ± 0.15 (`eligible`).

In [ ]:
pinn = rx.pinn_family_table(RUNS)
print("Sin corridas de PINN en runs/." if pinn.empty else f"{len(pinn)} corridas de PINN.")
pinn.round(2)

In [ ]:
rx.plume_to_puff(RUNS)

## 2. Las búsquedas

Cada búsqueda es una curva de pruebas (error con estación oculta de cada una y la mejor hasta ahora) frente a la corrida sin ajustar. Hay que leerla con tres preguntas: ¿el óptimo está **dentro** del espacio explorado o pegado a un borde (entonces falta ampliarlo)?, ¿cuánto se gana frente a no ajustar?, ¿qué parámetros importan (los que mueven el error)? El error de la búsqueda es algo optimista: usa validación simple, no anidada.

In [ ]:
table, _ = compare.collect(report.run_dirs(RUNS))
summary = report.tuning_summary(RUNS, table)
print("\n".join(rx.tuning_sentences(summary)) or "Sin búsquedas en runs/.")
summary

In [ ]:
report.plot_trials(RUNS, table)

### `pinn_puff.transport`

In [ ]:
transport = rx.read_search(RUNS, "pinn_puff.transport")
rx.search_overview(transport)

In [ ]:
rx.plot_trial_params(transport, "pinn_puff.transport")

### `pinn_puff.kernel`

In [ ]:
kernel = rx.read_search(RUNS, "pinn_puff.kernel")
rx.search_overview(kernel)

In [ ]:
rx.plot_trial_params(kernel, "pinn_puff.kernel")

## 3. La difusividad `kappa`

`kappa` es la difusividad turbulenta por unidad de velocidad del viento (`K = kappa · |u|`, en metros); el valor de la investigación es 1,500 m. Cada valor entrena su red (ADR 0004), así que la rejilla `pinn_puff_k*` es lo más caro. Se leen el error, el cociente domingo ÷ hábil con su intervalo (franja: tráfico) y el centro del placebo (franja: 1 ± 0.15).

In [ ]:
kappas = rx.kappa_runs(RUNS)
print("Sin corridas de la rejilla de kappa en runs/." if kappas.empty else f"{len(kappas)} valores de kappa.")
kappas.round(2)

In [ ]:
rx.plot_kappa(kappas)

## 4. Cómo se lee todo junto

1. **Un error menor no basta.** Una corrida con menor error con estación oculta pero con un placebo fuera de 1 ± 0.15 **no mejora el modelo**: aprendió algo que no es emisión. La tabla de la sección 1 y la figura de `kappa` muestran las dos cosas a la vez.
2. **El cociente domingo ÷ hábil debe moverse poco** con parámetros que no cambian la física de fondo (`tau_h`, `u_min`). Si una búsqueda lo mueve mucho para ganar décimas de ppb, el parámetro está absorbiendo otra cosa.
3. **`kappa` cambia el nivel absoluto, no los cocientes** (ADR 0004): si el cociente domingo ÷ hábil se mantiene en toda la rejilla, la conclusión sobre el cambio de emisión no depende de ese parámetro.
4. **Las ganancias hay que compararlas con el ruido.** El error con estación oculta cambia ±0.5 ppb solo por qué estaciones se usen (cuaderno 4); una ganancia de décimas de ppb entre pruebas no se distingue de ese ruido. La comparación con el cambio de transporte (pluma → puffs) está en el cuaderno 5.